# Data Quality

Este notebook valida a qualidade da base de NPS de um Ecommerce.

## Contexto da Base

Antes de avançar para a análise exploratória, esta etapa tem como objetivo verificar se a base está adequada para responder ao problema de negócio. A base contém informações de pedidos, entrega, atendimento, reclamações, recompra e satisfação dos clientes de um e-commerce.

A variável central do projeto é `nps_score`, que representa a nota de satisfação do cliente após a jornada de compra. Como essa nota será usada para classificar clientes entre detratores, neutros e promotores, é importante validar se os dados estão consistentes antes de qualquer análise.

Nesta etapa são avaliados pontos como valores ausentes, duplicidades, tipos de dados, escala do NPS, valores financeiros negativos, possíveis inconsistências e outliers. O objetivo não é alterar agressivamente a base, mas garantir que ela esteja confiável para a análise de negócio.


**Importação das bibliotecas necessárias para análise**


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

**Carregamento da base original**




In [2]:
data_nps = ("../data/raw/desafio_nps_fase_1.csv")
nps_ecom = pd.read_csv(data_nps)

# 1. Inspeção inicial na Base de Dados

Analisando as **primeiras** e as **últimas** linhas do dataset para conhecer as colunas e dados, vemos que os valores mantém uma consistência de acordo com o tipo de dado de cada coluna:

In [3]:
# Primeiras linhas do dataset
nps_ecom.head()

,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.9,0,3,6.5
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.4,0,3,0.0
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.8,0,7,1.5
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.9,0,4,0.3
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.1,0,3,7.9


In [4]:
# Últimas linhas do dataset
nps_ecom.tail()

,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
2495,2496,51,Sul,96,52496,615.81,6,11.41,2,14,3,28.96,2,1,2,3.7,0,3,4.3
2496,2497,37,Sul,89,52497,73.03,1,36.44,3,12,2,27.42,2,2,7,3.7,0,4,2.5
2497,2498,19,Sudeste,98,52498,522.78,1,4.84,9,2,2,38.94,1,1,1,7.4,0,3,6.2
2498,2499,41,Sul,51,52499,55.87,2,2.11,2,14,5,29.10,3,3,0,2.3,0,5,1.7
2499,2500,35,Nordeste,109,52500,420.94,1,16.51,4,11,1,34.16,3,1,0,10.0,1,2,7.6


Analisando os **tipos de dados** de cada coluna:

In [5]:
nps_ecom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer_id                2500 non-null   int64  
 1   customer_age               2500 non-null   int64  
 2   customer_region            2500 non-null   object 
 3   customer_tenure_months     2500 non-null   int64  
 4   order_id                   2500 non-null   int64  
 5   order_value                2500 non-null   float64
 6   items_quantity             2500 non-null   int64  
 7   discount_value             2500 non-null   float64
 8   payment_installments       2500 non-null   int64  
 9   delivery_time_days         2500 non-null   int64  
 10  delivery_delay_days        2500 non-null   int64  
 11  freight_value              2500 non-null   float64
 12  delivery_attempts          2500 non-null   int64  
 13  customer_service_contacts  2500 non-null   int64

Analisando o resumo **estatístico** de cada coluna:

In [6]:
nps_ecom.describe().round(2)

,customer_id,customer_age,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
count,2500.00,2500.00,2500.00,2500.00,2500.00,2500.00,2500.00,2500.00,2500.00,2500.00,2500.00,2500.00,2500.00,2500.00,2500.00,2500.00,2500.00,2500.00
mean,1250.50,43.40,61.32,51250.50,434.26,3.47,29.75,6.00,8.02,2.19,38.22,2.01,1.52,5.49,4.38,0.09,4.15,2.94
std,721.83,14.89,34.48,721.83,289.77,1.69,29.23,3.16,3.77,1.45,12.08,0.82,1.23,3.46,2.51,0.28,1.78,2.38
min,1.00,18.00,1.00,50001.00,7.76,1.00,0.02,1.00,2.00,0.00,2.62,1.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,625.75,31.00,31.00,50625.75,220.24,2.00,8.88,3.00,5.00,1.00,29.93,1.00,1.00,2.00,2.60,0.00,3.00,0.70
50%,1250.50,43.00,62.00,51250.50,375.52,3.00,20.94,6.00,8.00,2.00,38.50,2.00,1.00,6.00,4.40,0.00,4.00,2.80
75%,1875.25,56.00,91.00,51875.25,577.29,5.00,40.83,9.00,11.00,3.00,46.27,3.00,2.00,8.00,6.10,0.00,5.00,4.80
max,2500.00,69.00,119.00,52500.00,1983.81,6.00,230.33,11.00,14.00,8.00,76.13,3.00,7.00,11.00,10.00,1.00,11.00,10.00


Tamanho do dataset:

In [7]:
nps_ecom.shape

(2500, 19)

Avaliando quantos **valores distintos** tem em cada coluna:

In [8]:
nps_ecom.nunique()

customer_id                  2500
customer_age                   52
customer_region                 5
customer_tenure_months        119
order_id                     2500
order_value                  2457
items_quantity                  6
discount_value               2050
payment_installments           11
delivery_time_days             13
delivery_delay_days             9
freight_value                1897
delivery_attempts               3
customer_service_contacts       8
resolution_time_days           12
nps_score                     101
repeat_purchase_30d             2
complaints_count               12
csat_internal_score            98
dtype: int64

# 2. Consultando valores nulos

Analisando a média de **valores nulos** vemos que nessa base não tem nenhum.

In [9]:
nps_ecom.isnull().mean()

customer_id                  0.0
customer_age                 0.0
customer_region              0.0
customer_tenure_months       0.0
order_id                     0.0
order_value                  0.0
items_quantity               0.0
discount_value               0.0
payment_installments         0.0
delivery_time_days           0.0
delivery_delay_days          0.0
freight_value                0.0
delivery_attempts            0.0
customer_service_contacts    0.0
resolution_time_days         0.0
nps_score                    0.0
repeat_purchase_30d          0.0
complaints_count             0.0
csat_internal_score          0.0
dtype: float64

Analisando a média de **valores ausentes** vemos que nessa base não tem nenhum.

In [10]:
nps_ecom.isna().mean()

customer_id                  0.0
customer_age                 0.0
customer_region              0.0
customer_tenure_months       0.0
order_id                     0.0
order_value                  0.0
items_quantity               0.0
discount_value               0.0
payment_installments         0.0
delivery_time_days           0.0
delivery_delay_days          0.0
freight_value                0.0
delivery_attempts            0.0
customer_service_contacts    0.0
resolution_time_days         0.0
nps_score                    0.0
repeat_purchase_30d          0.0
complaints_count             0.0
csat_internal_score          0.0
dtype: float64

# 3. Checagens de qualidade

Esse bloco de *quality checks* cria uma tabela de validação para identificar possíveis problemas básicos na base antes da análise exploratória.

Primeiro, é criada uma lista vazia (`quality_checks`), onde cada checagem será armazenada. Em seguida, a função auxiliar `add_check()` adiciona uma nova linha nessa lista, registrando três informações: o item verificado, o resultado da checagem e um detalhe quantitativo.

Cada chamada da função valida um aspecto diferente da base, como valores ausentes, pedidos duplicados, NPS fora da escala esperada, idades fora de uma faixa plausível, valores financeiros negativos e inconsistências em atrasos, contatos ou reclamações. Ao final, a lista é convertida em um DataFrame (`quality_report`) para facilitar a leitura dos resultados em formato de tabela.

In [11]:
quality_checks = []

def add_check(item, result, detail):
    quality_checks.append({
        'checagem': item,
        'resultado': result,
        'detalhe': detail
    })

add_check(
    'Valores ausentes',
    'OK' if nps_ecom.isna().sum().sum() == 0 else 'Atenção',
    f"{nps_ecom.isna().sum().sum()} valores ausentes"
)

add_check(
    'Pedidos duplicados',
    'OK' if nps_ecom['order_id'].duplicated().sum() == 0 else 'Atenção',
    f"{nps_ecom['order_id'].duplicated().sum()} order_id duplicados"
)

add_check(
    'Clientes duplicados',
    'Info',
    f"{nps_ecom['customer_id'].duplicated().sum()} customer_id duplicados"
)

add_check(
    'NPS fora de 0 a 10',
    'OK' if nps_ecom['nps_score'].between(0, 10).all() else 'Atenção',
    f"{(~nps_ecom['nps_score'].between(0, 10)).sum()} registros fora da faixa"
)

add_check(
    'Idade fora de faixa plausível',
    'OK' if nps_ecom['customer_age'].between(18, 100).all() else 'Atenção',
    f"{(~nps_ecom['customer_age'].between(18, 100)).sum()} registros fora de 18-100"
)

add_check(
    'Valores financeiros negativos',
    'OK' if (nps_ecom[['order_value', 'discount_value', 'freight_value']] < 0).sum().sum() == 0 else 'Atenção',
    f"{(nps_ecom[['order_value', 'discount_value', 'freight_value']] < 0).sum().sum()} valores negativos"
)

add_check(
    'Atraso de entrega negativo',
    'OK' if (nps_ecom['delivery_delay_days'] < 0).sum() == 0 else 'Atenção',
    f"{(nps_ecom['delivery_delay_days'] < 0).sum()} atrasos negativos"
)

add_check(
    'Contatos/reclamações negativos',
    'OK' if (nps_ecom[['customer_service_contacts', 'complaints_count']] < 0).sum().sum() == 0 else 'Atenção',
    f"{(nps_ecom[['customer_service_contacts', 'complaints_count']] < 0).sum().sum()} valores negativos"
)

quality_report = pd.DataFrame(quality_checks)
display(quality_report)


,checagem,resultado,detalhe
0,Valores ausentes,OK,0 valores ausentes
1,Pedidos duplicados,OK,0 order_id duplicados
2,Clientes duplicados,Info,0 customer_id duplicados
3,NPS fora de 0 a 10,OK,0 registros fora da faixa
4,Idade fora de faixa plausível,OK,0 registros fora de 18-100
5,Valores financeiros negativos,OK,0 valores negativos
6,Atraso de entrega negativo,OK,0 atrasos negativos
7,Contatos/reclamações negativos,OK,0 valores negativos


# 4. Estatísticas descritivas

Esse bloco gera estatísticas descritivas para todas as colunas numéricas da base, permitindo uma visão geral da distribuição dos dados.

Primeiro, são selecionadas apenas as variáveis numéricas por meio de `select_dtypes(include='number')`. Em seguida, o método `describe()` calcula medidas como quantidade de registros, média, desvio padrão, valores mínimo e máximo, além de percentis específicos, como 1%, 5%, 25%, 50%, 75%, 95% e 99%.

A transposição da tabela com `.T` facilita a leitura, deixando cada variável em uma linha, enquanto o `.round(2)` arredonda os resultados para duas casas decimais. Essa etapa ajuda a identificar valores extremos, possíveis inconsistências e padrões gerais antes de aprofundar a análise exploratória.




In [12]:
num_cols = nps_ecom.select_dtypes(include='number').columns

descriptive_stats = (
    nps_ecom[num_cols]
    .describe(percentiles=[.01, .05, .25, .50, .75, .95, .99])
    .T
    .round(2)
)

display(descriptive_stats)


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
customer_id,2500.0,1250.50,721.83,1.00,25.99,125.95,625.75,1250.50,1875.25,2375.05,2475.01,2500.00
customer_age,2500.0,43.40,14.89,18.00,18.00,20.00,31.00,43.00,56.00,67.00,69.00,69.00
customer_tenure_months,2500.0,61.32,34.48,1.00,2.00,7.00,31.00,62.00,91.00,114.00,118.00,119.00
order_id,2500.0,51250.50,721.83,50001.00,50025.99,50125.95,50625.75,51250.50,51875.25,52375.05,52475.01,52500.00
order_value,2500.0,434.26,289.77,7.76,39.86,87.73,220.24,375.52,577.29,1013.78,1365.34,1983.81
items_quantity,2500.0,3.47,1.69,1.00,1.00,1.00,2.00,3.00,5.00,6.00,6.00,6.00
discount_value,2500.0,29.75,29.23,0.02,0.23,1.34,8.88,20.94,40.83,88.09,133.67,230.33
payment_installments,2500.0,6.00,3.16,1.00,1.00,1.00,3.00,6.00,9.00,11.00,11.00,11.00
delivery_time_days,2500.0,8.02,3.77,2.00,2.00,2.00,5.00,8.00,11.00,14.00,14.00,14.00
delivery_delay_days,2500.0,2.19,1.45,0.00,0.00,0.00,1.00,2.00,3.00,5.00,6.00,8.00


# 5. Limpeza e criação de variaveis

In [13]:
# Criando uma cópia da base original para preservar nps_ecom
nps_clean = nps_ecom.copy()

# Padronizando os nomes das colunas
nps_clean.columns = nps_clean.columns.str.strip().str.lower()

# Removendo linhas totalmente duplicadas
nps_clean = nps_clean.drop_duplicates().copy()

# Criando a classificação de NPS
conditions = [
    nps_clean['nps_score'] <= 6,
    (nps_clean['nps_score'] > 6) & (nps_clean['nps_score'] < 9),
    nps_clean['nps_score'] >= 9
]

choices = ['Detrator', 'Neutro', 'Promotor']

nps_clean['nps_group'] = np.select(
    conditions,
    choices,
    default='Indefinido'
)

# Criando flags binárias para facilitar análises e cálculos
nps_clean['is_detractor'] = (nps_clean['nps_score'] <= 6).astype(int)
nps_clean['is_promoter'] = (nps_clean['nps_score'] >= 9).astype(int)

# Criando faixas de atraso na entrega
nps_clean['delay_bucket'] = pd.cut(
    nps_clean['delivery_delay_days'],
    bins=[-1, 0, 1, 2, 3, 5, np.inf],
    labels=[
        'Sem atraso',
        '1 dia',
        '2 dias',
        '3 dias',
        '4-5 dias',
        'Acima de 5 dias'
    ]
)

# Criando faixas de quantidade de reclamações
nps_clean['complaints_bucket'] = pd.cut(
    nps_clean['complaints_count'],
    bins=[-1, 0, 1, 2, 3, 5, np.inf],
    labels=[
        '0',
        '1',
        '2',
        '3',
        '4-5',
        'Acima de 5'
    ]
)

# Criando faixas de contatos com atendimento
nps_clean['contacts_bucket'] = pd.cut(
    nps_clean['customer_service_contacts'],
    bins=[-1, 0, 1, 2, 3, np.inf],
    labels=[
        '0',
        '1',
        '2',
        '3',
        'Acima de 3'
    ]
)

# Criando faixas de CSAT interno
nps_clean['csat_bucket'] = pd.cut(
    nps_clean['csat_internal_score'],
    bins=[-0.1, 2, 4, 6, 8, 10],
    labels=[
        '0-2',
        '2-4',
        '4-6',
        '6-8',
        '8-10'
    ]
)

# Visualizando a base tratada
display(nps_clean.head())

print(f'Linhas após limpeza: {nps_clean.shape[0]:,}')
print(f'Colunas após preparação: {nps_clean.shape[1]:,}')

,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,...,repeat_purchase_30d,complaints_count,csat_internal_score,nps_group,is_detractor,is_promoter,delay_bucket,complaints_bucket,contacts_bucket,csat_bucket
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,...,0,3,6.5,Neutro,0,0,2 dias,3,0,6-8
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,...,0,3,0.0,Detrator,1,0,4-5 dias,3,0,0-2
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,...,0,7,1.5,Detrator,1,0,1 dia,Acima de 5,Acima de 3,0-2
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,...,0,4,0.3,Detrator,1,0,2 dias,4-5,1,0-2
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,...,0,3,7.9,Neutro,0,0,1 dia,3,1,6-8


Linhas após limpeza: 2,500
Colunas após preparação: 26


**Salvando a base tratada**


In [14]:
from pathlib import Path

PROCESSED_DATA_PATH = Path('../data/processed/desafio_nps_fase_1_clean.csv')

if not PROCESSED_DATA_PATH.parent.exists():
    PROCESSED_DATA_PATH = Path('data/processed/desafio_nps_fase_1_clean.csv')

PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

nps_clean.to_csv(
    PROCESSED_DATA_PATH,
    index=False,
    encoding='utf-8'
)

print(f'Base tratada salva em: {PROCESSED_DATA_PATH}')


Base tratada salva em: ..\data\processed\desafio_nps_fase_1_clean.csv


# Conclusão desta etapa

A base está consistente e pronta para a análise exploratória. Durante a validação, não foram encontrados problemas relevantes de qualidade dos dados, como valores nulos, notas de NPS fora da escala esperada ou valores financeiros negativos. Como preparação para as análises, foram criadas algumas variáveis derivadas para facilitar a interpretação dos resultados sob a ótica do negócio, com destaque para a classificação dos clientes em detratores, neutros e promotores.
